# Agentic RAG Finance Demo with Multiple Tools

**Business Problem:**  
We need an AI-driven Retrieval-Augmented Generation (RAG) system to answer finance queries with source citations, leveraging large financial documents.

**Data Files Used available in folder ./financial_docs:**  
- `Q1_2024_Financial_Report.txt`  
- `Market_Analysis_2024.txt`  

**Program Flow:**  
1. Load, split, embed, and index documents in Chroma.  
2. Define RAG and summarization tools using Cohere LLM.  
3. Initialize a Zero-Shot agent with multiple tools.  
4. Provide a CLI interface for interactive querying.  

**Agentic RAG Theory:**  
The agent uses LangChain’s RetrievalQA to fetch relevant document chunks via embeddings, then a Cohere LLM to generate answers with citations, wrapped in a tool. A Zero-Shot React Description agent selects the correct tool per query.

## Import Libraries

In [ ]:
import os
from langchain_cohere import ChatCohere
from langchain.document_loaders import TextLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_cohere import CohereEmbeddings
from langchain_chroma import Chroma
from langchain.chains import RetrievalQA
from langchain.agents import Tool, initialize_agent, AgentType

## Load, Split, Embed, and Index Documents

In [ ]:
import oci
from LoadProperties import LoadProperties
properties=LoadProperties()

# Embedding Model
from langchain_community.embeddings import OCIGenAIEmbeddings
embeddings = OCIGenAIEmbeddings(
    model_id=properties.getEmbeddingModelName(),
    service_endpoint=properties.getEndpoint(),
    compartment_id=properties.getCompartment(),
    auth_type='INSTANCE_PRINCIPAL',
)

# COHERE_API_KEY=""
# embeddings = CohereEmbeddings(cohere_api_key=COHERE_API_KEY)

# source document folder
# Files in folder "Market_Analysis_2024.txt", "Q1_2024_Financial_Report.txt"

docs_folder = "./financial_docs"  

# Load all .txt files
loaders = [TextLoader(os.path.join(docs_folder, f), encoding="utf8")
           for f in os.listdir(docs_folder) if f.endswith(".txt")]

all_docs = []
for loader in loaders:
    all_docs.extend(loader.load())

splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.split_documents(all_docs)

vectorstore = Chroma.from_documents(chunks, embeddings, persist_directory="./chromadb")

### Build a RetrievalQA chain 

In [ ]:

from langchain_community.chat_models.oci_generative_ai import ChatOCIGenAI

# Initialize the Cohere language model
llm = ChatOCIGenAI(
      model_id='meta.llama-3.3-70b-instruct',
      service_endpoint=properties.getEndpoint(),
      compartment_id=properties.getCompartment(),auth_type='INSTANCE_PRINCIPAL',
      model_kwargs={ "max_tokens": 600},)

# llm = ChatCohere( model = 'command-a-03-2025',cohere_api_key=COHERE_API_KEY,temperature=0)


qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=vectorstore.as_retriever(search_kwargs={"k":3}),
    return_source_documents=True
)

def financial_qa(query: str) -> str:
    """Tool function: runs QA and appends source citations."""
    result = qa_chain({"query": query})
    answer = result["result"]
    sources = result["source_documents"]
    citation_lines = []
    for doc in sources:
        src = os.path.basename(doc.metadata.get("source", "unknown"))
        citation_lines.append(f"- {src} (page chunk)")
    citations = "\n".join(citation_lines)
    return f"{answer}\n\nSources:\n{citations}"

## Build a summarization chain and tool functions

In [ ]:

from langchain.prompts import PromptTemplate
from langchain.chains import LLMChain

summary_prompt = PromptTemplate(
    input_variables=["text"],
    template=(
        "You are a finance-domain expert. "
        "Provide a concise summary of the following document:\n\n{text}"        
    )
)
summary_chain = LLMChain(llm=llm, prompt=summary_prompt)

def list_documents(_: str) -> str:
    """Lists all source files in the docs folder."""
    files = os.listdir(docs_folder)
    return "Available documents:\n" + "\n".join(f"- {f}" for f in files)

def summarize_financial_report(_: str) -> str:
    """Summarizes the Q1 2024 financial report."""
    path = os.path.join(docs_folder, "Q1_2024_Financial_Report.txt")
    with open(path, encoding="utf8") as f:
        content = f.read()
    return summary_chain.run(text=content)

def summarize_market_analysis(_: str) -> str:
    """Summarizes the 2024 market analysis report."""
    path = os.path.join(docs_folder, "Market_Analysis_2024.txt")
    with open(path, encoding="utf8") as f:
        content = f.read()
    return summary_chain.run(text=content)

# ── Wrap everything in an agent ─────────────────────────────────────────────
tools = [
    Tool(
        name="FinancialRAG",
        func=financial_qa,
        description=(
            "Answer detailed finance questions about Q1 2024 "
            "and provide source citations."
        )
    ),
    Tool(
        name="ListDocuments",
        func=list_documents,
        description="List all available source documents."
    ),
    Tool(
        name="SummarizeFinancialReport",
        func=summarize_financial_report,
        description="Provide a concise summary of the Q1 2024 financial report.Used only when user explicitly requests summary on Finance report"
    ),
    Tool(
        name="SummarizeMarketAnalysis",
        func=summarize_market_analysis,
        description="Provide a concise summary of the 2024 market analysis report.Used only when user explicitly requests summary on market analysis report"
    ),
]


agent = initialize_agent(
    tools,
    llm,
    agent=AgentType.ZERO_SHOT_REACT_DESCRIPTION, handle_parsing_errors=True,
    verbose=True
)

## Pass a query to the Agent 
### Query1

In [ ]:
# query = "Summarize our net income performance in Q1 2024."
query = "SummarizeFinancialReport"

response = agent.invoke(query)
print("\n=== Agent Response ===\n")

from IPython.display import Markdown, display
print(response['input'],"\n")
display((Markdown(response['output'])))

### Query2

In [ ]:

query = "What was our YoY revenue growth in Q1 2024?"

response = agent.invoke(query)
print("\n=== Agent Response ===\n")

from IPython.display import Markdown, display
print(response['input'],"\n")
display((Markdown(response['output'])))


### Query3

In [ ]:

# query = "According to the Market_Analysis_2024 report, what were the key market trends observed in Q1 2024?"
query = "SummarizeMarketAnalysis"
response = agent.invoke(query)
print("\n=== Agent Response ===\n")

from IPython.display import Markdown, display
print(response['input'],"\n")
display((Markdown(response['output'])))

### Query4

In [ ]:
query = "What was our YoY revenue growth in Q1 2024, and what market volatility trends were noted in the Market_Analysis_2024 report?"

response = agent.invoke(query)
print("\n=== Agent Response ===\n")

from IPython.display import Markdown, display
print(response['input'],"\n")
display((Markdown(response['output'])))

### Query5

In [ ]:
query = "ListDocuments"

response = agent.invoke(query)
print("\n=== Agent Response ===\n")

from IPython.display import Markdown, display
print(response['input'],"\n")
display((Markdown(response['output'])))

### Query6

In [ ]:
query = "Summarize our net income and then list the available documents."

response = agent.invoke(query)
print("\n=== Agent Response ===\n")

from IPython.display import Markdown, display
print(response['input'],"\n")
display((Markdown(response['output'])))